In [1]:
# ============================================================
# FINAL MODELS — CALIBRATION / BRIER SUMMARY
# PCA TRAIN-ONLY FINAL VERSION
# ============================================================

import numpy as np
import pandas as pd
from pathlib import Path

from sklearn.metrics import (
    brier_score_loss,
    roc_auc_score,
    average_precision_score,
    log_loss
)

# ------------------------------------------------------------
# Paths
# ------------------------------------------------------------

PREDICTION_FILES = {
    "victimization": Path("./final_victim/DT_victim_PCA_trainonly_TEST/predictions_with_probs.csv"),
    "perpetration": Path("./final_perpetrator/content/perpetrator_v3/predictions_with_probs.csv"),
    "overlap": Path(
        "./final_overlap/overlap_final/"
        "overlap_LOGREG_SW_pos1p5_PCA_trainonly_FINAL_PCA095_thr0p5_seed42/"
        "outputs/predictions_with_probs.csv"
    ),
}

OUTPUT_DIR = Path("./final_calibration_brier_PCA_trainonly")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

N_BINS = 10


# ------------------------------------------------------------
# Helpers
# ------------------------------------------------------------

def detect_col(df, candidates):
    cols_lower = {str(c).lower(): c for c in df.columns}

    for cand in candidates:
        if cand.lower() in cols_lower:
            return cols_lower[cand.lower()]

    for c in df.columns:
        cl = str(c).lower()
        if any(cand.lower() in cl for cand in candidates):
            return c

    return None


def load_prediction_file(path, outcome):
    df = pd.read_csv(path)

    y_true_col = detect_col(df, [
        f"y_true_{outcome}",
        "y_true_victim",
        "y_true_perp",
        "y_true_perpetration",
        "y_true_overlap",
        "y_true_intersect",
        "y_true",
        "target"
    ])

    y_pred_col = detect_col(df, [
        f"y_pred_{outcome}",
        "y_pred_victim",
        "y_pred_perp",
        "y_pred_perpetration",
        "y_pred_overlap",
        "y_pred_intersect",
        "y_pred",
        "prediction"
    ])

    y_prob_col = detect_col(df, [
        f"y_prob_{outcome}",
        "y_prob_victim",
        "y_prob_perp",
        "y_prob_perpetration",
        "y_prob_overlap",
        "y_prob_intersect",
        "y_prob",
        "probability",
        "score"
    ])

    if y_true_col is None or y_pred_col is None or y_prob_col is None:
        raise ValueError(
            f"Could not detect required columns for {outcome}. "
            f"Columns found: {list(df.columns)}"
        )

    out = pd.DataFrame({
        "y_true": pd.to_numeric(df[y_true_col], errors="coerce"),
        "y_pred": pd.to_numeric(df[y_pred_col], errors="coerce"),
        "y_prob": pd.to_numeric(df[y_prob_col], errors="coerce"),
    }).dropna()

    out["y_true"] = (out["y_true"] > 0).astype(int)
    out["y_pred"] = (out["y_pred"] > 0).astype(int)

    # Clip probabilities for log loss safety
    out["y_prob"] = out["y_prob"].clip(1e-6, 1 - 1e-6)

    return out


def calibration_bins(df, outcome, n_bins=10):
    d = df.copy()

    # Equal-width bins 0-1
    d["prob_bin"] = pd.cut(
        d["y_prob"],
        bins=np.linspace(0, 1, n_bins + 1),
        include_lowest=True,
        duplicates="drop"
    )

    rows = []

    for bin_label, sub in d.groupby("prob_bin", observed=False):
        if len(sub) == 0:
            continue

        rows.append({
            "outcome": outcome,
            "bin": str(bin_label),
            "n": int(len(sub)),
            "mean_predicted_probability": float(sub["y_prob"].mean()),
            "observed_event_rate": float(sub["y_true"].mean()),
            "absolute_calibration_error": float(abs(sub["y_prob"].mean() - sub["y_true"].mean())),
            "positives": int(sub["y_true"].sum()),
            "negatives": int((sub["y_true"] == 0).sum()),
        })

    return pd.DataFrame(rows)


# ------------------------------------------------------------
# Run calibration/Brier
# ------------------------------------------------------------

summary_rows = []
all_bins = []

for outcome, path in PREDICTION_FILES.items():
    print("\n======================================")
    print("Outcome:", outcome)
    print("Path:", path)
    print("Exists:", path.exists())

    dfp = load_prediction_file(path, outcome)

    y_true = dfp["y_true"].to_numpy()
    y_prob = dfp["y_prob"].to_numpy()
    y_pred = dfp["y_pred"].to_numpy()

    prevalence = y_true.mean()

    brier = brier_score_loss(y_true, y_prob)

    try:
        auc_roc = roc_auc_score(y_true, y_prob)
    except Exception:
        auc_roc = np.nan

    try:
        auc_pr = average_precision_score(y_true, y_prob)
    except Exception:
        auc_pr = np.nan

    try:
        ll = log_loss(y_true, y_prob, labels=[0, 1])
    except Exception:
        ll = np.nan

    mean_prob_pos = y_prob[y_true == 1].mean()
    mean_prob_neg = y_prob[y_true == 0].mean()

    # Brier skill score against prevalence-only model
    # Reference Brier = prevalence * (1 - prevalence)
    brier_ref = prevalence * (1 - prevalence)
    brier_skill = 1 - (brier / brier_ref) if brier_ref > 0 else np.nan

    summary_rows.append({
        "outcome": outcome,
        "n": int(len(y_true)),
        "positives": int(y_true.sum()),
        "negatives": int((y_true == 0).sum()),
        "prevalence": prevalence,
        "mean_predicted_probability": float(y_prob.mean()),
        "mean_probability_positives": float(mean_prob_pos),
        "mean_probability_negatives": float(mean_prob_neg),
        "brier_score": float(brier),
        "brier_reference_prevalence_model": float(brier_ref),
        "brier_skill_score": float(brier_skill),
        "roc_auc": float(auc_roc),
        "average_precision_pr_auc": float(auc_pr),
        "log_loss": float(ll),
    })

    bins_df = calibration_bins(dfp, outcome, n_bins=N_BINS)
    all_bins.append(bins_df)

    print("n:", len(y_true))
    print("prevalence:", round(prevalence, 3))
    print("mean predicted probability:", round(y_prob.mean(), 3))
    print("Brier:", round(brier, 4))
    print("Brier skill:", round(brier_skill, 4))
    print("ROC AUC:", round(auc_roc, 4))
    print("PR AUC:", round(auc_pr, 4))


summary_df = pd.DataFrame(summary_rows)
bins_all_df = pd.concat(all_bins, ignore_index=True)

summary_df.to_csv(OUTPUT_DIR / "final_models_brier_calibration_summary.csv", index=False)
bins_all_df.to_csv(OUTPUT_DIR / "final_models_calibration_bins.csv", index=False)

print("\n=== BRIER / CALIBRATION SUMMARY ===")
print(summary_df.to_string(index=False, float_format=lambda x: f"{x:.4f}"))

print("\n=== CALIBRATION BINS ===")
print(bins_all_df.to_string(index=False, float_format=lambda x: f"{x:.4f}"))

print("\nSaved to:")
print(OUTPUT_DIR.resolve())


Outcome: victimization
Path: final_victim/DT_victim_PCA_trainonly_TEST/predictions_with_probs.csv
Exists: True
n: 942
prevalence: 0.494
mean predicted probability: 0.497
Brier: 0.2307
Brier skill: 0.0771
ROC AUC: 0.6342
PR AUC: 0.5881

Outcome: perpetration
Path: final_perpetrator/content/perpetrator_v3/predictions_with_probs.csv
Exists: True
n: 942
prevalence: 0.235
mean predicted probability: 0.575
Brier: 0.2802
Brier skill: -0.5607
ROC AUC: 0.6951
PR AUC: 0.3985

Outcome: overlap
Path: final_overlap/overlap_final/overlap_LOGREG_SW_pos1p5_PCA_trainonly_FINAL_PCA095_thr0p5_seed42/outputs/predictions_with_probs.csv
Exists: True
n: 942
prevalence: 0.189
mean predicted probability: 0.528
Brier: 0.2506
Brier skill: -0.6355
ROC AUC: 0.7635
PR AUC: 0.4361

=== BRIER / CALIBRATION SUMMARY ===
      outcome   n  positives  negatives  prevalence  mean_predicted_probability  mean_probability_positives  mean_probability_negatives  brier_score  brier_reference_prevalence_model  brier_skill_score

In [2]:
from pathlib import Path
import numpy as np
import pandas as pd
import joblib

from sklearn.metrics import (
    confusion_matrix,
    roc_auc_score,
    average_precision_score,
    brier_score_loss,
)

BASE = Path("/Users/legna/WORKSPACE/ICREA/app_overlap/results_repo/intersection_model/Version_Final")

ART_BASE = BASE / "final_model_artifacts_for_S13"

OUTDIR = BASE / "final_calibration_brier_PCA_trainonly_RECALC_FROM_FROZEN_ARTIFACTS"
OUTDIR.mkdir(parents=True, exist_ok=True)

THRESHOLDS = {
    "victimization": 0.500,
    "perpetration": 0.500,
    "overlap": 0.523164,
}

EXPECTED = {
    "victimization": {"TP": 405, "FP": 329, "TN": 148, "FN": 60},
    "perpetration": {"TP": 202, "FP": 504, "TN": 217, "FN": 19},
    "overlap": {"TP": 144, "FP": 355, "TN": 409, "FN": 34},
}

SOURCE_FEATURE_MATRIX_PATHS = {
    "victimization": BASE / "final_victim/df_victima_feat.csv",
    "perpetration": BASE / "final_perpetrator/content/perpetrator_v3/df_perpetrador_feat.csv",
    "overlap": BASE / "final_perpetrator/content/perpetrator_v3/df_perpetrador_feat.csv",
}

ALIGNED_PATHS = {
    "victimization": BASE / "aligned_test_exports/victimization_aligned_test_rowlevel.csv",
    "perpetration": BASE / "aligned_test_exports/perpetration_aligned_test_rowlevel.csv",
    "overlap": BASE / "aligned_test_exports/overlap_aligned_test_rowlevel.csv",
}

ARTIFACTS = {
    "victimization": {
        "scaler": ART_BASE / "victimization/scaler.joblib",
        "pca": ART_BASE / "victimization/pca.joblib",
        "model": ART_BASE / "victimization/model.joblib",
        "feature_names": ART_BASE / "victimization/feature_names.csv",
        "means": ART_BASE / "victimization/means.csv",
    },
    "perpetration": {
        "scaler": ART_BASE / "perpetration/scaler.joblib",
        "pca": ART_BASE / "perpetration/pca.joblib",
        "model": ART_BASE / "perpetration/model.keras",
        "feature_names": ART_BASE / "perpetration/feature_names.csv",
        "means": ART_BASE / "perpetration/means.csv",
    },
    "overlap": {
        "scaler": ART_BASE / "overlap/scaler.joblib",
        "pca": ART_BASE / "overlap/pca.joblib",
        "model": ART_BASE / "overlap/model.joblib",
        "feature_names": ART_BASE / "overlap/feature_names.csv",
        "means": ART_BASE / "overlap/means.csv",
    },
}

alias_map = {
    "PAIS": "PAÍS",
    "PAÍS": "PAIS",
    "CONVIVEN.5": "CONVIVEN_H",
    "CONVIVEN.6": "CONVIVEN_0",
    "CONVIVEN_H": "CONVIVEN.5",
    "CONVIVEN_0": "CONVIVEN.6",
    "GENERO_BIN_0": "GENERO.BN0",
    "GENERO_BIN_1": "GENERO.BN1",
    "GENERO.BN0": "GENERO_BIN_0",
    "GENERO.BN1": "GENERO_BIN_1",
    "ORIENTSEX.BN_1": "ORIENTSEX.BN0",
    "ORIENTSEX.BN_2": "ORIENTSEX.BN1",
    "ORIENTSEX.BN0": "ORIENTSEX.BN_1",
    "ORIENTSEX.BN1": "ORIENTSEX.BN_2",
}

def ensure_alias_columns(df):
    df = df.copy()
    for wanted, alt in alias_map.items():
        if wanted not in df.columns and alt in df.columns:
            df[wanted] = df[alt]
    return df

def load_feature_names(path):
    return pd.read_csv(path, header=None).iloc[:, 0].astype(str).tolist()

def load_means(path):
    df = pd.read_csv(path)
    if "mean" in df.columns:
        return df["mean"].astype(float).values
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    if numeric_cols:
        return df[numeric_cols[-1]].astype(float).values
    raise ValueError(f"No numeric mean column found in {path}")

def load_model(path):
    path = Path(path)
    if path.suffix == ".keras":
        from tensorflow.keras.models import load_model as keras_load_model
        return keras_load_model(path)
    return joblib.load(path)

def model_n_features(model):
    if hasattr(model, "n_features_in_"):
        return int(model.n_features_in_)
    if hasattr(model, "input_shape"):
        return int(model.input_shape[-1])
    return None

def get_score(model, X):
    if hasattr(model, "predict_proba"):
        proba = model.predict_proba(X)
        classes = list(getattr(model, "classes_", [0, 1]))
        col = classes.index(1) if 1 in classes else 1
        return proba[:, col].astype(float)

    pred = model.predict(X, verbose=0) if "verbose" in model.predict.__code__.co_varnames else model.predict(X)
    return np.asarray(pred).ravel().astype(float)

def get_y(aligned):
    for c in ["y_true_official", "y_true", "y_true_overlap"]:
        if c in aligned.columns:
            return aligned[c].astype(int).values
    raise ValueError("No y_true column found.")

rows = []

for outcome in ["victimization", "perpetration", "overlap"]:
    print("\n" + "=" * 90)
    print("Outcome:", outcome)

    art = ARTIFACTS[outcome]

    for k, p in art.items():
        assert Path(p).exists(), f"Missing {outcome} artifact {k}: {p}"

    aligned_path = ALIGNED_PATHS[outcome]
    assert aligned_path.exists(), f"Missing aligned file: {aligned_path}"

    src_path = SOURCE_FEATURE_MATRIX_PATHS[outcome]
    assert src_path.exists(), f"Missing source feature matrix: {src_path}"

    aligned = pd.read_csv(aligned_path)
    src = pd.read_csv(src_path)
    src = ensure_alias_columns(src)

    y = get_y(aligned)

    idx_col = None
    for c in ["idx_original", "filtered_idx", "row_id", "original_index", "index"]:
        if c in aligned.columns:
            idx_col = c
            break

    if idx_col is None:
        raise ValueError(f"No index column found in {aligned_path}")

    idx = aligned[idx_col].astype(int).values

    feature_names = load_feature_names(art["feature_names"])
    missing = [c for c in feature_names if c not in src.columns]
    if missing:
        raise ValueError(f"{outcome}: missing features in source matrix: {missing}")

    X_raw = src.iloc[idx][feature_names].copy()

    scaler = joblib.load(art["scaler"])
    pca = joblib.load(art["pca"])
    model = load_model(art["model"])
    means = load_means(art["means"])

    X_scaled = scaler.transform(X_raw)
    X_centered = X_scaled - means
    X_pca_full = pca.transform(X_centered)

    n_req = model_n_features(model)
    if n_req is None:
        X_model = X_pca_full
    else:
        X_model = X_pca_full[:, :n_req]

    score = get_score(model, X_model)

    threshold = THRESHOLDS[outcome]
    pred = (score >= threshold).astype(int)

    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel()

    prevalence = float(np.mean(y))
    brier = float(brier_score_loss(y, score))
    brier_ref = float(prevalence * (1 - prevalence))
    brier_skill = float(1 - brier / brier_ref)
    roc_auc = float(roc_auc_score(y, score))
    pr_auc = float(average_precision_score(y, score))
    mean_pred = float(np.mean(score))

    print("n:", len(y))
    print("positives:", int(y.sum()))
    print("threshold:", threshold)
    print("TP FP TN FN:", tp, fp, tn, fn)
    print("mean_pred:", mean_pred)
    print("prevalence:", prevalence)
    print("brier:", brier)
    print("brier_ref:", brier_ref)
    print("brier_skill:", brier_skill)
    print("roc_auc:", roc_auc)
    print("pr_auc:", pr_auc)

    exp = EXPECTED[outcome]
    matches = (
        tp == exp["TP"]
        and fp == exp["FP"]
        and tn == exp["TN"]
        and fn == exp["FN"]
    )

    print("matches official operating point:", matches)

    if not matches:
        raise RuntimeError(f"{outcome} does not reproduce official operating point.")

    rows.append({
        "outcome": outcome,
        "n": len(y),
        "positives": int(y.sum()),
        "negatives": int(len(y) - y.sum()),
        "prevalence": prevalence,
        "mean_predicted_probability": mean_pred,
        "brier_score": brier,
        "brier_reference_prevalence_model": brier_ref,
        "brier_skill_score": brier_skill,
        "roc_auc": roc_auc,
        "average_precision_pr_auc": pr_auc,
        "threshold_for_validation": threshold,
        "TP": tp,
        "FP": fp,
        "TN": tn,
        "FN": fn,
        "matches_official_operating_point": matches,
    })

summary = pd.DataFrame(rows)

out_path = OUTDIR / "final_models_brier_calibration_summary_RECALC_FROM_FROZEN_ARTIFACTS.csv"
summary.to_csv(out_path, index=False)

print("\n" + "=" * 90)
print("SAVED:", out_path)
print(summary.to_string(index=False))


Outcome: victimization
n: 942
positives: 465
threshold: 0.5
TP FP TN FN: 405 329 148 60
mean_pred: 0.49720090244289433
prevalence: 0.49363057324840764
brier: 0.23067550393751896
brier_ref: 0.2499594304028561
brier_skill: 0.07714822535103993
roc_auc: 0.6342012127769888
pr_auc: 0.5880607408346172
matches official operating point: True

Outcome: perpetration


/Users/legna/WORKSPACE/ICREA/app_overlap/results_repo/intersection_model/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2820: UserWarning: X has feature names, but MinMaxScaler was fitted without feature names
  warnings.warn(


30/30 ━━━━━━━━━━━━━━━━━━━━ 0s 689us/step
n: 942
positives: 221
threshold: 0.5
TP FP TN FN: 202 504 217 19
mean_pred: 0.5747039369337118
prevalence: 0.2346072186836518
brier: 0.2802442866314283
brier_ref: 0.17956667162517298
brier_skill: -0.5606698286217029
roc_auc: 0.6950690657144113
pr_auc: 0.3984508426195649
matches official operating point: True

Outcome: overlap
n: 942
positives: 178
threshold: 0.523164
TP FP TN FN: 144 355 409 34
mean_pred: 0.551835355267334
prevalence: 0.18895966029723993
brier: 0.2686257576994871
brier_ref: 0.15325390707759162
brier_skill: -0.7528150689396997
roc_auc: 0.7536252132478382
pr_auc: 0.4335309108027502
matches official operating point: True

SAVED: /Users/legna/WORKSPACE/ICREA/app_overlap/results_repo/intersection_model/Version_Final/final_calibration_brier_PCA_trainonly_RECALC_FROM_FROZEN_ARTIFACTS/final_models_brier_calibration_summary_RECALC_FROM_FROZEN_ARTIFACTS.csv
      outcome   n  positives  negatives  prevalence  mean_predicted_probability  b

/Users/legna/WORKSPACE/ICREA/app_overlap/results_repo/intersection_model/.venv/lib/python3.12/site-packages/keras/src/saving/saving_lib.py:798: UserWarning: Skipping variable loading for optimizer 'rmsprop', because it has 8 variables whereas the saved optimizer has 14 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))
/Users/legna/WORKSPACE/ICREA/app_overlap/results_repo/intersection_model/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2820: UserWarning: X has feature names, but MinMaxScaler was fitted without feature names
  warnings.warn(
